In [ ]:
free -h 
nvidia-smi

In [ ]:
 
 source activate MyEnvForSD
 conda install -c conda-forge colmap

In [ ]:
# need to install cudaversion colmap pycolmap

In [ ]:
pip install pycolmap-cuda12 --index-url https://pypi.org/simple

In [ ]:
pip install pycolmap-cuda12 --index-url https://pypi.org/simple

In [ ]:
git clone https://github.com/graphdeco-inria/gaussian-splatting.git
cd gaussian-splatting
git submodule update --init --recursive --depth 1
cd gaussian-splatting
sorce activate MyEnvForSD
pip install /home/aistudio/models/gaussian-splatting/submodules/simple-knn --no-build-isolation
pip install /home/aistudio/models/gaussian-splatting/submodules/diff-gaussian-rasterization --no-build-isolation

# test

In [1]:
import torch
import diff_gaussian_rasterization
import simple_knn

print('✅ Successfully imported 3DGS submodules')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ CUDA is not detected by PyTorch.')

✅ Successfully imported 3DGS submodules
CUDA available: True
GPU: Tesla V100-SXM2-32GB


In [ ]:
# Move the .bin files from sparse/0 to distorted/sparse/0
mv /home/aistudio/models/ColmapOutputs/Watt/sparse/0/*.bin /home/aistudio/models/ColmapOutputs/Watt/distorted/sparse/0/
# Create a symbolic link named 'input' inside the Watt folder
ln -s /home/aistudio/models/realityscan/Watt_Resized /home/aistudio/models/ColmapOutputs/Watt/input

In [ ]:
 pip install plyfile
 

![Colmap output folder ](colmapout_folder_hier.jpg)



# create colmap folder strucutres

In [1]:
# 1. 创建核心目录
export PROJECT_PATH=/home/aistudio/models/realityscan/MyBicycles
mkdir -p $PROJECT_PATH/input/images
mkdir -p $PROJECT_PATH/dist/sparse

# 2. 检查路径
echo "请确保你的 100 张照片已存放在: $PROJECT_PATH/input/images"

SyntaxError: invalid syntax (3269294210.py, line 2)

In [ ]:
# 特征提取

# need to set use_gpu 0  for cpu,cannot use gpu for some reason
colmap feature_extractor \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --ImageReader.single_camera 0 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 0 \
    --SiftExtraction.num_threads 32

# 穷举匹配 (100张图大约需要 10 分钟)
colmap exhaustive_matcher \
    --database_path $PROJECT_PATH/dist/database.db \
    --SiftMatching.use_gpu 0 \
    --SiftMatching.num_threads 32

# 稀疏重建

colmap mapper \
    --database_path $PROJECT_PATH/dist/database.db \
    --image_path $PROJECT_PATH/input/images \
    --output_path $PROJECT_PATH/dist/sparse

In [ ]:
import pycolmap
from pathlib import Path

# 定义路径
project_path = Path("/home/aistudio/models/realityscan/MyBicycles")
database_path = project_path / "dist/database.db"
image_path = project_path / "input/images"

# 1. 特征提取 (可以使用 GPU)
# PyCOLMAP 内部处理了无头环境的上下文，通常能直接调起 CUDA
pycolmap.extract_features(
    database_path=database_path,
    image_path=image_path,
    camera_model="PINHOLE",
    # 如果 GPU 依然不可用，PyCOLMAP 会自动退回到 CPU，
    # 或者你可以手动指定 SiftExtractionOptions
)

# 2. 穷举匹配
pycolmap.match_exhaustive(database_path=database_path)

# 3. 增量式重建 (对应 colmap mapper)
output_path = project_path / "dist/sparse"
output_path.mkdir(parents=True, exist_ok=True)

reconstructions = pycolmap.incremental_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=output_path
)

print(f"重建完成！共生成 {len(reconstructions)} 个模型。")

In [ ]:
python train.py -s /home/aistudio/models/ColmapOutputs/Watt --model_path /home/aistudio/models/ColmapOutputs/Watt/output

In [ ]:
colmap model_analyzer --path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 

In [ ]:
# mothod1 works fine need add sparse/0
# use image undistorter to undistort images  gen  undistorted folder

colmap image_undistorter \
    --image_path /home/aistudio/models/ColmapOutputs/MyBicycles/images \
    --input_path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 \
    --output_path /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
    --output_type COLMAP 
#colmap output_path  /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/ 
# but 3dgs train need to create a folder 0 and move the file to the subfolder 0

mkdir -p /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0

mv /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/*.bin \
   /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/

# mothod2 works fine
# the output folder 
#python /home/aistudio/models/gaussian-splatting/convert.py \
#    -s /home/aistudio/models/ColmapOutputs/MyBicycles
    
python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
-m /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/output \
--save_iterations 7000 14000 21000 28000 30000
#  first will convert point3d.bin to point3d.ply
# then train the model in 30000 steps every 7000 steps save a model
 

In [ ]:
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/gaussian-splatting$ pip install plyfile
Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
Collecting plyfile
  Downloading http://mirrors.baidubce.com/pypi/packages/22/22/1755bb4c7db15bb1ed63b4eb7a7fc133bf42a3f9cc806c0d5941e107ba90/plyfile-1.1.3-py3-none-any.whl (36 kB)
Requirement already satisfied: numpy>=1.21 in /home/aistudio/external-libraries/lib/python3.10/site-packages (from plyfile) (2.2.6)
Installing collected packages: plyfile
Successfully installed plyfile-1.1.3
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/gaussian-splatting$ python train.py -s /home/aistudio/models/ColmapOutputs/Watt --model_path /home/aistudio/models/ColmapOutputs/Watt/output
Optimizing /home/aistudio/models/ColmapOutputs/Watt/output
Output folder: /home/aistudio/models/ColmapOutputs/Watt/output [29/03 18:46:52]
Tensorboard not available: not logging progress [29/03 18:46:52]
Reading camera 415/415 [29/03 18:46:54]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [29/03 18:46:54]
Loading Training Cameras [29/03 18:46:55]
Loading Test Cameras [29/03 18:47:35]
Number of points at initialisation :  108160 [29/03 18:47:35]
Training progress:  23%|█████████████▎                                           | 7000/30000 [05:08<21:14, 18.05it/s, Loss=0.1032912, Depth Loss=0.0000000]
[ITER 7000] Evaluating train: L1 0.05458241552114487 PSNR 21.37511329650879 [29/03 18:52:45]

[ITER 7000] Saving Gaussians [29/03 18:52:45]
Training progress: 100%|████████████████████████████████████████████████████████| 30000/30000 [28:52<00:00, 17.31it/s, Loss=0.0770907, Depth Loss=0.0000000]

[ITER 30000] Evaluating train: L1 0.044078940898180013 PSNR 22.83071517944336 [29/03 19:16:29]

[ITER 30000] Saving Gaussians [29/03 19:16:29]

Training complete. [29/03 19:16:52]

# use superspl.at to edit upload 

In [ ]:
https://superspl.at/ 
https://superspl.at/scene/1f9f0dd2/edit

# MyBicycles training logs


In [ ]:
aistudio@jupyter-2563417-10053501:~$ source activate MyEnvForSD
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~$ cd /home/aistudio/models/ColmapOutputs/MyBicycles
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ ls - al
ls: cannot access '-': No such file or directory
ls: cannot access 'al': No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ ls -al
total 28
drwxr-xr-x 1 aistudio aistudio 4096 Apr  2 09:52 .
drwxr-xr-x 1 aistudio aistudio 4096 Apr  1 11:59 ..
drwxr-xr-x 2 aistudio aistudio 4096 Apr  1 11:59 .ipynb_checkpoints
drwxr-xr-x 2 aistudio aistudio 4096 Apr  2 09:52 images
drwxr-xr-x 3 aistudio aistudio 4096 Apr  2 09:51 sparse
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python train.py -s MyBicyclesmodels/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s MyBicyclesmodels/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
Optimizing 
Output folder: ./output/885b6d10-a [02/04 10:11:37]
Tensorboard not available: not logging progress [02/04 10:11:37]
Reading camera 1/100Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 194, in readColmapSceneInfo
    cam_infos_unsorted = readColmapCameras(
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 98, in readColmapCameras
    assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"
AssertionError: Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ colmap image_undistorter \
>     --image_path /home/aistudio/models/ColmapOutputs/MyBicycles/images \
>     --input_path /home/aistudio/models/ColmapOutputs/MyBicycles/sparse/0 \
>     --output_path /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted \
>     --output_type COLMAP

==============================================================================
Reading reconstruction
==============================================================================

 => Reconstruction with 100 images and 52934 points

==============================================================================
Image undistortion
==============================================================================

Undistorting image [1/100]
Undistorting image [2/100]
Undistorting image [3/100]
Undistorting image [4/100]
Undistorting image [5/100]
Undistorting image [6/100]
Undistorting image [7/100]
Undistorting image [8/100]
Undistorting image [9/100]
Undistorting image [10/100]
Undistorting image [11/100]
Undistorting image [12/100]
Undistorting image [13/100]
Undistorting image [14/100]
Undistorting image [15/100]
Undistorting image [16/100]
Undistorting image [17/100]
Undistorting image [18/100]
Undistorting image [19/100]
Undistorting image [20/100]
Undistorting image [21/100]
Undistorting image [22/100]
Undistorting image [23/100]
Undistorting image [24/100]
Undistorting image [25/100]
Undistorting image [26/100]
Undistorting image [27/100]
Undistorting image [28/100]
Undistorting image [29/100]
Undistorting image [30/100]
Undistorting image [31/100]
Undistorting image [32/100]
Undistorting image [33/100]
Undistorting image [34/100]
Undistorting image [35/100]
Undistorting image [36/100]
Undistorting image [37/100]
Undistorting image [38/100]
Undistorting image [39/100]
Undistorting image [40/100]
Undistorting image [41/100]
Undistorting image [42/100]
Undistorting image [43/100]
Undistorting image [44/100]
Undistorting image [45/100]
Undistorting image [46/100]
Undistorting image [47/100]
Undistorting image [48/100]
Undistorting image [49/100]
Undistorting image [50/100]
Undistorting image [51/100]
Undistorting image [52/100]
Undistorting image [53/100]
Undistorting image [54/100]
Undistorting image [55/100]
Undistorting image [56/100]
Undistorting image [57/100]
Undistorting image [58/100]
Undistorting image [59/100]
Undistorting image [60/100]
Undistorting image [61/100]
Undistorting image [62/100]
Undistorting image [63/100]
Undistorting image [64/100]
Undistorting image [65/100]
Undistorting image [66/100]
Undistorting image [67/100]
Undistorting image [68/100]
Undistorting image [69/100]
Undistorting image [70/100]
Undistorting image [71/100]
Undistorting image [72/100]
Undistorting image [73/100]
Undistorting image [74/100]
Undistorting image [75/100]
Undistorting image [76/100]
Undistorting image [77/100]
Undistorting image [78/100]
Undistorting image [79/100]
Undistorting image [80/100]
Undistorting image [81/100]
Undistorting image [82/100]
Undistorting image [83/100]
Undistorting image [84/100]
Undistorting image [85/100]
Undistorting image [86/100]
Undistorting image [87/100]
Undistorting image [88/100]
Undistorting image [89/100]
Undistorting image [90/100]
Undistorting image [91/100]
Undistorting image [92/100]
Undistorting image [93/100]
Undistorting image [94/100]
Undistorting image [95/100]
Undistorting image [96/100]
Undistorting image [97/100]
Undistorting image [98/100]
Undistorting image [99/100]
Undistorting image [100/100]
Writing reconstruction...
Writing configuration...
Writing scripts...
Elapsed time: 0.432 [minutes]
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles
Optimizing 
Output folder: ./output/9c374f8e-d [02/04 10:14:23]
Tensorboard not available: not logging progress [02/04 10:14:23]
Reading camera 1/100Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 194, in readColmapSceneInfo
    cam_infos_unsorted = readColmapCameras(
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 98, in readColmapCameras
    assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"
AssertionError: Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted
Optimizing 
Output folder: ./output/fbb6bb37-2 [02/04 10:15:57]
Tensorboard not available: not logging progress [02/04 10:15:57]
Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 149, in readColmapSceneInfo
    cam_extrinsics = read_extrinsics_binary(cameras_extrinsic_file)
  File "/home/aistudio/models/gaussian-splatting/scene/colmap_loader.py", line 187, in read_extrinsics_binary
    with open(path_to_model_file, "rb") as fid:
FileNotFoundError: [Errno 2] No such file or directory: '/home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/images.bin'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/aistudio/models/gaussian-splatting/train.py", line 282, in <module>
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)
  File "/home/aistudio/models/gaussian-splatting/train.py", line 51, in training
    scene = Scene(dataset, gaussians)
  File "/home/aistudio/models/gaussian-splatting/scene/__init__.py", line 44, in __init__
    scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.depths, args.eval, args.train_test_exp)
  File "/home/aistudio/models/gaussian-splatting/scene/dataset_readers.py", line 154, in readColmapSceneInfo
    cam_extrinsics = read_extrinsics_text(cameras_extrinsic_file)
  File "/home/aistudio/models/gaussian-splatting/scene/colmap_loader.py", line 249, in read_extrinsics_text
    with open(path, "r") as fid:
FileNotFoundError: [Errno 2] No such file or directory: '/home/aistudio/models/ColmapOutputs/MyBicycles/undistorted/sparse/0/images.txt'
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ python models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/distorted
python: can't open file '/home/aistudio/models/ColmapOutputs/MyBicycles/models/gaussian-splatting/train.py': [Errno 2] No such file or directory
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$  python /home/aistudio/models/gaussian-splatting/train.py -s /home/aistudio/models/ColmapOutputs/MyBicycles/undistorted
Optimizing 
Output folder: ./output/2fa449bd-b [02/04 10:17:47]
Tensorboard not available: not logging progress [02/04 10:17:47]
Reading camera 100/100 [02/04 10:17:47]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [02/04 10:17:47]
Loading Training Cameras [02/04 10:17:47]
Loading Test Cameras [02/04 10:18:47]
Number of points at initialisation :  52934 [02/04 10:18:48]
Training progress:  23%|███████████▏                                    | 7000/30000 [15:15<46:53,  8.17it/s, Loss=0.1162505, Depth Loss=0.0000000]
[ITER 7000] Evaluating train: L1 0.05141039192676544 PSNR 21.697405242919924 [02/04 10:34:06]

[ITER 7000] Saving Gaussians [02/04 10:34:06]
Training progress: 100%|█████████████████████████████████████████████| 30000/30000 [1:10:29<00:00,  7.09it/s, Loss=0.0923409, Depth Loss=0.0000000]^A

[ITER 30000] Evaluating train: L1 0.02892332747578621 PSNR 26.950869750976565 [02/04 11:29:20]

[ITER 30000] Saving Gaussians [02/04 11:29:20]
Killed
(MyEnvForSD) aistudio@jupyter-2563417-10053501:~/models/ColmapOutputs/MyBicycles$ 